# Bài tập 03 - Naive Bayes

Notebook được chạy từ thư mục gốc `03. Naive_Bayes`.

Trên Colab, hãy mount Drive rồi dùng `%cd` để đi đến thư mục này trước khi chọn **Run all**. Hai tệp dữ liệu phải nằm tại `data/tin_nhan.csv` và `data/sinh_vien.csv`.


## Bài 1


In [1]:
"""Bài 1: áp dụng định lý Bayes cho ba nhóm sinh viên."""

priors = {"Nhóm 1": 0.50, "Nhóm 2": 0.30, "Nhóm 3": 0.20}
pass_rates = {"Nhóm 1": 0.95, "Nhóm 2": 0.70, "Nhóm 3": 0.30}

evidence = sum(priors[group] * pass_rates[group] for group in priors)
posteriors = {
    group: priors[group] * pass_rates[group] / evidence for group in priors
}

print(f"P(qua môn) = {evidence:.4f}")
for group, probability in posteriors.items():
    print(f"P({group} | qua môn) = {probability:.4f}")
print(f"Tổng ba xác suất hậu nghiệm = {sum(posteriors.values()):.4f}")
print("Giải thích: nhóm 1 chiếm tỷ lệ ban đầu lớn và có xác suất qua môn cao nhất,")
print("nên một sinh viên đã qua môn có khả năng thuộc nhóm 1 cao nhất.")


P(qua môn) = 0.7450
P(Nhóm 1 | qua môn) = 0.6376
P(Nhóm 2 | qua môn) = 0.2819
P(Nhóm 3 | qua môn) = 0.0805
Tổng ba xác suất hậu nghiệm = 1.0000
Giải thích: nhóm 1 chiếm tỷ lệ ban đầu lớn và có xác suất qua môn cao nhất,
nên một sinh viên đã qua môn có khả năng thuộc nhóm 1 cao nhất.


## Bài 2


In [2]:
"""Bài 2: tìm các từ đặc trưng nhất cho hai lớp."""

from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "tin_nhan.csv")
vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X = vectorizer.fit_transform(df["noi_dung"])
y = df["la_rac"]

model = MultinomialNB(alpha=1.0).fit(X, y)
words = vectorizer.get_feature_names_out()
log_ratio = model.feature_log_prob_[1] - model.feature_log_prob_[0]

spam_indices = log_ratio.argsort()[::-1][:5]
normal_indices = log_ratio.argsort()[:5]

print("5 từ gợi ý thư rác mạnh nhất:")
for index in spam_indices:
    print(f"  {words[index]:<16} log-ratio = {log_ratio[index]:.4f}")

print("\n5 từ gợi ý tin thường mạnh nhất:")
for index in normal_indices:
    print(f"  {words[index]:<16} log-ratio = {log_ratio[index]:.4f}")

print("\nLog-ratio dương lớn nghiêng về lớp rác; âm lớn về độ lớn nghiêng về lớp thường.")


5 từ gợi ý thư rác mạnh nhất:
  mãi              log-ratio = 3.2034
  tay              log-ratio = 2.9056
  ngay             log-ratio = 2.6618
  miễn             log-ratio = 2.4795
  trúng            log-ratio = 2.3742

5 từ gợi ý tin thường mạnh nhất:
  lớp              log-ratio = -2.6122
  em               log-ratio = -2.5618
  bài              log-ratio = -2.2946
  báo              log-ratio = -2.2741
  phòng            log-ratio = -2.2741

Log-ratio dương lớn nghiêng về lớp rác; âm lớn về độ lớn nghiêng về lớp thường.


## Bài 3


In [3]:
"""Bài 3: khảo sát ảnh hưởng của hệ số làm trơn alpha."""

from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "tin_nhan.csv")
texts_train, texts_test, y_train, y_test = train_test_split(
    df["noi_dung"], df["la_rac"], test_size=0.25, random_state=17,
    stratify=df["la_rac"]
)
vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

print(f"{'alpha':>8} {'accuracy':>10} {'precision':>10} {'recall':>10}")
for alpha in [0.001, 0.1, 1, 10, 100]:
    model = MultinomialNB(alpha=alpha).fit(X_train, y_train)
    prediction = model.predict(X_test)
    print(
        f"{alpha:>8g} {accuracy_score(y_test, prediction):>10.4f} "
        f"{precision_score(y_test, prediction):>10.4f} "
        f"{recall_score(y_test, prediction):>10.4f}"
    )

print("\nAlpha quá lớn kéo xác suất của các từ về gần nhau, làm giảm sức phân biệt của mô hình.")


   alpha   accuracy  precision     recall
   0.001     0.9467     1.0000     0.8667
     0.1     0.9467     1.0000     0.8667
       1     0.9467     1.0000     0.8667
      10     0.9467     1.0000     0.8667
     100     0.9467     1.0000     0.8667

Alpha quá lớn kéo xác suất của các từ về gần nhau, làm giảm sức phân biệt của mô hình.


## Bài 4


In [4]:
"""Bài 4: tự cài đặt Multinomial Naive Bayes và so sánh sklearn."""

from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "tin_nhan.csv")
texts_train, texts_test, y_train, y_test = train_test_split(
    df["noi_dung"], df["la_rac"], test_size=0.25, random_state=17,
    stratify=df["la_rac"]
)

# Chỉ học từ điển trên tập huấn luyện để tránh rò rỉ dữ liệu.
vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)
vocabulary = vectorizer.get_feature_names_out()
word_to_index = {word: index for index, word in enumerate(vocabulary)}


class ManualMultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, texts, labels):
        labels = np.asarray(labels)
        self.classes_ = np.unique(labels)
        self.class_log_prior_ = {}
        self.feature_log_prob_ = {}
        for label in self.classes_:
            class_texts = texts[labels == label]
            counts = Counter(word for text in class_texts for word in text.split())
            total = sum(counts[word] for word in vocabulary)
            denominator = total + self.alpha * len(vocabulary)
            self.class_log_prior_[label] = np.log(len(class_texts) / len(texts))
            self.feature_log_prob_[label] = np.array([
                np.log((counts[word] + self.alpha) / denominator)
                for word in vocabulary
            ])
        return self

    def predict(self, texts):
        predictions = []
        for text in texts:
            counts = Counter(text.split())
            x = np.zeros(len(vocabulary))
            for word, count in counts.items():
                if word in word_to_index:
                    x[word_to_index[word]] = count
            scores = {
                label: self.class_log_prior_[label] + x @ self.feature_log_prob_[label]
                for label in self.classes_
            }
            predictions.append(max(scores, key=scores.get))
        return np.asarray(predictions)


manual = ManualMultinomialNB(alpha=1.0).fit(
    texts_train.to_numpy(), y_train.to_numpy()
)
manual_prediction = manual.predict(texts_test.to_numpy())

sklearn_model = MultinomialNB(alpha=1.0).fit(X_train, y_train)
sklearn_prediction = sklearn_model.predict(X_test)

print(f"Số từ học từ tập train: {len(vocabulary)}")
print(f"Accuracy tự cài đặt: {accuracy_score(y_test, manual_prediction):.4f}")
print(f"Accuracy sklearn   : {accuracy_score(y_test, sklearn_prediction):.4f}")
print(f"Số dự đoán giống nhau: {(manual_prediction == sklearn_prediction).sum()}/{len(y_test)}")
print("\nDùng log giúp biến phép nhân nhiều xác suất nhỏ thành phép cộng,")
print("tránh hiện tượng số quá nhỏ bị làm tròn về 0 (underflow).")


Số từ học từ tập train: 40
Accuracy tự cài đặt: 0.9467
Accuracy sklearn   : 0.9467
Số dự đoán giống nhau: 75/75

Dùng log giúp biến phép nhân nhiều xác suất nhỏ thành phép cộng,
tránh hiện tượng số quá nhỏ bị làm tròn về 0 (underflow).


## Bài 5


In [5]:
"""Bài 5: so sánh MultinomialNB và GaussianNB trên dữ liệu liên tục."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

multinomial = MultinomialNB(alpha=1.0).fit(X_train, y_train)
gaussian = GaussianNB().fit(X_train, y_train)

print(f"Accuracy MultinomialNB: {accuracy_score(y_test, multinomial.predict(X_test)):.4f}")
print(f"Accuracy GaussianNB   : {accuracy_score(y_test, gaussian.predict(X_test)):.4f}")
print("\nGaussianNB phù hợp hơn vì giờ ôn và điểm giữa kỳ là biến liên tục.")
print("MultinomialNB được thiết kế cho số đếm/rời rạc, nên giả định của nó không phù hợp ở đây.")


Accuracy MultinomialNB: 0.8000
Accuracy GaussianNB   : 0.9000

GaussianNB phù hợp hơn vì giờ ôn và điểm giữa kỳ là biến liên tục.
MultinomialNB được thiết kế cho số đếm/rời rạc, nên giả định của nó không phù hợp ở đây.


## Bài 6


In [6]:
"""Bài 6: đánh giá mô hình khi dữ liệu bị mất cân bằng."""

from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "tin_nhan.csv")
normal = df[df["la_rac"] == 0]
spam = df[df["la_rac"] == 1].head(20)
imbalanced = pd.concat([normal, spam], ignore_index=True)

texts_train, texts_test, y_train, y_test = train_test_split(
    imbalanced["noi_dung"], imbalanced["la_rac"], test_size=0.25,
    random_state=17, stratify=imbalanced["la_rac"]
)
vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)
model = MultinomialNB(alpha=1.0).fit(X_train, y_train)
prediction = model.predict(X_test)

print("Phân bố dữ liệu mất cân bằng:")
print(imbalanced["la_rac"].value_counts().sort_index().to_string())
print("\nMa trận nhầm lẫn:")
print(confusion_matrix(y_test, prediction))
print(f"Accuracy      : {accuracy_score(y_test, prediction):.4f}")
print(f"Spam precision: {precision_score(y_test, prediction, zero_division=0):.4f}")
print(f"Spam recall   : {recall_score(y_test, prediction, zero_division=0):.4f}")
print("\nAccuracy có thể vẫn cao vì lớp thường chiếm đa số.")
print("Spam recall quan trọng hơn để biết mô hình đã bỏ sót bao nhiêu tin rác.")


Phân bố dữ liệu mất cân bằng:
la_rac
0    180
1     20

Ma trận nhầm lẫn:
[[45  0]
 [ 1  4]]
Accuracy      : 0.9800
Spam precision: 1.0000
Spam recall   : 0.8000

Accuracy có thể vẫn cao vì lớp thường chiếm đa số.
Spam recall quan trọng hơn để biết mô hình đã bỏ sót bao nhiêu tin rác.
